# NYPD Reported Rape Complaints

a simple pandas analysis of reported rape complaints in the NYPD complaint dataset.

## 0. Setup

Run this first. We will use pandas only.

In [ ]:
import pandas as pd

data = ""

pd.set_option("display.max_columns", 80) #lets you see all the columns in the dataframe
pd.set_option("display.max_rows", 30)

## 1. Load The Data

Load the CSV into a dataframe called `df`. run basic analysis to understand how big you dataset is and what's in there

**Hints:**
- `pd.read_csv(...)`
- `df.shape`
- if you see Nan in some cells, it is not a text, it is an empty value. 

you can get rid of rows with empty values by using `df.dropna()`
drop columns with empty values df.dropna(axis=1)

BE CAREFUL! because once you drop them, they only way to go back is to reload the dataset from the start. 

that's why when you drop something, it's better to put it into a new variable like df_clean or df_new

In [ ]:
df = pd.read_csv()

## 2. Inspect The Columns

**Question:** Which columns look useful for analyzing time, place, offense type, victim/suspect groups, and reporting delay?

**Hints:**
- `df.columns`
- `df.head()` # to print the first 5 rows. also there is df.tail. or you can specify how much you want to see like df.head(100)

## 3. Find The Rape Offense Rows

**Question:** How many rows have `OFNS_DESC == "RAPE"`?

**Hints:**
- Boolean filters look like: `df[df["column"] == "value"]`
- To avoid capitalization problems: `df["OFNS_DESC"].str.upper().eq("RAPE")`
- Use `.copy()` after filtering if you plan to add new columns.

In [ ]:
# TODO: Filter to reported rape complaints.
rape = df[df["OFNS_DESC"].fillna("").str.upper().eq("RAPE")].copy()

# TODO: Count the rows.
len(rape)

## 4. Offense Details

**Question:** Within the rape category, what specific `PD_DESC` descriptions appear most often?

**Hints:**
- `value_counts()` counts categories.
- `normalize=True` gives proportions.
- Multiply by 100 and round to make percentages.

In [ ]:
# TODO: Count PD_DESC values.
rape["PD_DESC"].value_counts(dropna=False)

In [ ]:
# TODO: Convert those counts into percentages.
(rape["PD_DESC"].value_counts(normalize=True, dropna=False) * 100).round(1)

## 5. Parse Dates And Times

**Question:** What is the difference between report date and complaint start date in this dataset?

**Hints:**
- `CMPLNT_FR_DT` is the complaint start date.
- `RPT_DT` is the report date.
- Use `pd.to_datetime(..., errors="coerce")`.
- After parsing a datetime column, use `.dt.year`, `.dt.day_name()`, or `.dt.hour`.

In [ ]:
# TODO: Parse date and time columns.
rape["complaint_date"] = pd.to_datetime(rape["CMPLNT_FR_DT"], format="%m/%d/%Y", errors="coerce")
rape["report_date"] = pd.to_datetime(rape["RPT_DT"], format="%m/%d/%Y", errors="coerce")
rape["complaint_time"] = pd.to_datetime(rape["CMPLNT_FR_TM"], format="%H:%M:%S", errors="coerce")

# TODO: Create useful date/time columns.
rape["year"] = rape["complaint_date"].dt.year
rape["day_of_week"] = rape["complaint_date"].dt.day_name()
rape["hour"] = rape["complaint_time"].dt.hour

rape[["CMPLNT_FR_DT", "RPT_DT", "CMPLNT_FR_TM", "complaint_date", "report_date", "hour"]].head()

## 6. Separate Reported Rows From 2026 Occurrence Rows

The file is current year-to-date by report date, but some complaints were reported in 2026 for incidents that started earlier.

**Question:** How many rape complaints in the file have a complaint start date in 2026?

**Hints:**
- Count occurrence years with `value_counts().sort_index()`.
- Filter with `rape[rape["year"].eq(2026)]`.

In [ ]:
# TODO: Count complaint start years.
rape["year"].value_counts(dropna=False).sort_index()

In [ ]:
# TODO: Create a dataframe for rows whose complaint start date is in 2026.
rape_2026 = rape[rape["year"].eq(2026)].copy()

len(rape_2026)

## 7. Time Of Day

**Question:** What time of day do 2026 reported rape complaints most often start?

**Hints:**
- Use `pd.cut()` to group hours into periods.
- Example bins: `[0, 6, 12, 18, 24]`.
- Think carefully about `00:00:00` and `00:01:00`. These can be real times, but they can also be placeholders when exact time is unclear.

In [ ]:
time_labels = [
    "Overnight, 12am-5:59am",
    "Morning, 6am-11:59am",
    "Afternoon, 12pm-5:59pm",
    "Evening, 6pm-11:59pm",
]

# TODO: Create a time_period column.
rape_2026["time_period"] = pd.cut(
    rape_2026["hour"],
    bins=[0, 6, 12, 18, 24],
    labels=time_labels,
    right=False,
    include_lowest=True,
)

# TODO: Count time periods.
rape_2026["time_period"].value_counts().reindex(time_labels)

In [ ]:
# TODO: Repeat the time analysis after excluding exact 00:00 and 00:01.
not_midnight_placeholder = ~rape_2026["CMPLNT_FR_TM"].isin([
    "0:00:00", "00:00:00", "0:01:00", "00:01:00"
])

rape_2026_time_check = rape_2026[not_midnight_placeholder].copy()

rape_2026_time_check["time_period"].value_counts().reindex(time_labels)

## 8. Top Hours

**Question:** Which specific hours are most common after excluding exact `00:00` and `00:01`?

**Hints:**
- Use `value_counts().head(10)`.
- Use `.sort_index()` if you want hours in chronological order.

In [ ]:
# TODO: Count the most common hours.
rape_2026_time_check["hour"].value_counts().head(12)

## 9. Borough Patterns

**Question:** Which boroughs have the most 2026 rape complaint rows?

**Hints:**
- `value_counts()`
- `normalize=True`
- A count is not a rate. To make a true rate, you would need population data.

In [ ]:
# TODO: Count by borough.
rape_2026["BORO_NM"].value_counts(dropna=False)

In [ ]:
# TODO: Calculate borough percentages.
(rape_2026["BORO_NM"].value_counts(normalize=True, dropna=False) * 100).round(1)

## 10. Precincts As A Neighborhood Proxy

The CSV does not have a neighborhood-name column. With pandas only, use `ADDR_PCT_CD` plus borough as the closest built-in geography.

**Question:** Which precinct-borough combinations appear most often?

**Hints:**
- Create a new column by combining strings.
- `fillna("Unknown")` can keep missing values from breaking string operations.

In [ ]:
# TODO: Create a precinct_borough column.
rape_2026["precinct_borough"] = (
    rape_2026["ADDR_PCT_CD"].fillna("Unknown")
    + " - "
    + rape_2026["BORO_NM"].fillna("Unknown")
)

# TODO: Show the top 15 precinct-borough combinations.
rape_2026["precinct_borough"].value_counts().head(15)

## 11. Premises And Location Type

**Question:** Are complaints mostly linked to residences, streets, hotels, or other premises? Are they usually listed as inside or outside?

**Hints:**
- Use `PREM_TYP_DESC` for premises.
- Use `LOC_OF_OCCUR_DESC` for inside/front/rear/opposite information.

In [ ]:
# TODO: Count premises.
rape_2026["PREM_TYP_DESC"].value_counts(dropna=False).head(12)

In [ ]:
# TODO: Count location-of-occurrence descriptions.
rape_2026["LOC_OF_OCCUR_DESC"].value_counts(dropna=False)

## 12. Day Of Week

**Question:** Which day of week has the most 2026 complaint-start rows? Is the difference large or small?

**Hints:**
- You already created `day_of_week` before filtering.
- For readable order, use `.reindex([...])` with a list of weekday names.

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# TODO: Count day of week in calendar order.
rape_2026["day_of_week"].value_counts().reindex(weekday_order)

## 13. Victim And Suspect Age Groups

**Question:** What age groups appear most often for victims and suspects? Why should you be careful when interpreting these fields?

**Hints:**
- Use aggregate counts only.
- Missing or `UNKNOWN` values are meaningful data-quality signals.

In [ ]:
# TODO: Count victim age groups.
rape["VIC_AGE_GROUP"].value_counts(dropna=False)

In [ ]:
# TODO: Count suspect age groups.
rape["SUSP_AGE_GROUP"].value_counts(dropna=False)

## 14. Reporting Delay

**Question:** How long after the complaint start date were complaints reported?

**Hints:**
- Subtract datetime columns: `report_date - complaint_date`.
- Use `.dt.days` to convert a timedelta to number of days.
- Use `.describe()` for summary statistics.
- Compare same-day, within-7-days, and more-than-1-year reports.

In [ ]:
# TODO: Calculate report delay in days.
rape["report_delay_days"] = (rape["report_date"] - rape["complaint_date"]).dt.days

delay = rape["report_delay_days"].dropna()
delay.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).round(1)

In [ ]:
# TODO: Calculate reporting-delay shares.
same_day_share = (delay == 0).mean() * 100
within_7_days_share = (delay <= 7).mean() * 100
over_365_days_share = (delay > 365).mean() * 100

same_day_share, within_7_days_share, over_365_days_share

## 15. Make A Reusable Count Table Function

Now make the analysis cleaner by writing a small function.

**Question:** Can you create a table with both counts and percentages for any column?

**Hints:**
- A function starts with `def function_name(...):`.
- `series.value_counts(dropna=False)` gives counts.
- Build a dataframe from a dictionary: `pd.DataFrame({"count": counts, "percent": ...})`.

In [ ]:
# TODO: Fill in this function.
def count_table(series, total=None, top=10):
    if total is None:
        total = len(series)
    
    counts = series.value_counts(dropna=False).head(top)
    percentages = (counts / total * 100).round(1)
    
    return pd.DataFrame({"count": counts, "percent": percentages})


# Try it:
count_table(rape_2026["BORO_NM"])

## 16. Cross-Tab: Time Period By Borough

**Question:** Do time-of-day patterns look different by borough?

**Hints:**
- `pd.crosstab(index, columns)`
- Add `normalize="columns"` to compare percentages within each borough.
- Use `.round(1)` after multiplying by 100.

In [ ]:
# TODO: Count time periods by borough.
pd.crosstab(rape_2026["time_period"], rape_2026["BORO_NM"]).reindex(time_labels)

In [ ]:
# TODO: Show percentages within each borough.
(pd.crosstab(rape_2026["time_period"], rape_2026["BORO_NM"], normalize="columns") * 100).round(1).reindex(time_labels)

## 17. Harder: Compare Rape Complaints To All Complaints By Borough

**Question:** What share of all complaint rows in each borough are rape complaint rows? How is this different from saying a borough has a high crime rate?

**Hints:**
- Count all complaints by borough from `df`.
- Count rape complaints by borough from `rape` or `rape_2026`.
- Combine series with `pd.concat([...], axis=1)`.
- This is a share of complaint rows, not a population rate.

In [ ]:
# TODO: Count all complaint rows by borough.
all_by_borough = df["BORO_NM"].value_counts(dropna=False)

# TODO: Count rape complaint rows by borough.
rape_by_borough = rape["BORO_NM"].value_counts(dropna=False)

# TODO: Combine and calculate a percentage.
borough_compare = pd.concat(
    [all_by_borough, rape_by_borough],
    axis=1,
    keys=["all_complaints", "rape_complaints"],
).fillna(0)

borough_compare["rape_share_of_all_complaints_pct"] = (
    borough_compare["rape_complaints"] / borough_compare["all_complaints"] * 100
).round(2)

borough_compare.sort_values("rape_share_of_all_complaints_pct", ascending=False)

## 18. Write Your Findings

Answer these in plain language:

1. How many reported rape complaint rows are in the file?
2. How many have complaint start dates in 2026?
3. What are the most common time periods?
4. What happens to the time pattern when you exclude exact `00:00` and `00:01`?
5. Which boroughs and precincts appear most often?
6. What kinds of premises appear most often?
7. What data-quality caveats should a reader know?

**Writing hint:** Use words like "reported complaints" and "complaint rows," not "all incidents."

## 19. Optional Export

**Question:** Save one or two summary tables as CSV files for your story notes.

**Hints:**
- `table.to_csv("filename.csv")`
- Keep exports aggregate only.

In [ ]:
# TODO: Export aggregate tables if useful.
# count_table(rape_2026["BORO_NM"]).to_csv("rape_2026_by_borough.csv")
# count_table(rape_2026["precinct_borough"], top=20).to_csv("rape_2026_by_precinct.csv")